# 16 — ewm-cortex-fpga: bridge pipeline + LUT-view (end-to-end)

This notebook runs in the **ewm-cortex-fpga** workspace and demonstrates
everything implemented in the M4-fpga milestone:

1. **Soldered core is bit-exact** — bridge hashing and InLUT recovery vs the vendored `hllset-core`/`hllset-materialize`.
2. **The cortex black-box as an `ewm-dsl` pipeline** — `Slice → Gate` over the bridge's golden module registry.
3. **Bridge Phase D nodes** — dual-encoding `SliceModule`, exact-LUT `GateModule`, `GroundModule` τ/ρ report.
4. **LUT-view v1** — the content-addressed active-vocabulary cache, with refresh-iff-changed.


In [2]:
:dep cortex-fpga = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/cortex-fpga" }
:dep lut-view = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/lut-view" }
:dep cortex-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/cortex-core" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/hllset-core" }
:dep hllset-materialize = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex-fpga/crates/hllset-materialize" }
:dep ewm-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-fpga-bridge/crates/ewm-core" }
:dep ewm-modules = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-fpga-bridge/crates/ewm-modules" }
:dep ewm-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-fpga-bridge/crates/ewm-dsl" }


In [3]:
// All imports in one place (evcxr keeps a single scope).
use cortex_core::CortexPipeline;
use cortex_fpga::{cortex_pipeline, run_cortex_pipeline, view_from_tokens};
use ewm_core::{
    murmur3_hash, parse_token_id, slice_positions_with, token_in_bytes, token_in_bytes_le,
    token_to_position, InLut,
};
use ewm_modules::{GateModule, GroundModule, Module, Packet, SliceModule, Stream};
use hllset_core::HLLSet;
use hllset_core::core::hashing as vendored;
use hllset_materialize::{materialize_inlut, TokenLUT};
println!("crates loaded: cortex-fpga, lut-view, ewm-core, ewm-modules, vendored reference");


crates loaded: cortex-fpga, lut-view, ewm-core, ewm-modules, vendored reference


---
## 1. The soldered core is bit-exact

The bridge pins the HLLSet hashing contract (MurmurHash3, `P=10, M=1024, 32 bits/reg`).
The vendored `hllset-core` must agree bit-for-bit, or the whole bridge integration is invalid.


In [4]:
for token in ["tid0", "tid671", "tid18308", "hello"] {
    let bytes = token.as_bytes();
    let ours = (murmur3_hash(bytes), token_to_position(bytes));
    let theirs = (vendored::murmur3_hash(bytes), vendored::token_to_position(bytes));
    println!("{token:>10}: hash={} pos={:?}  matches={}", ours.0, ours.1, ours == theirs);
}
println!("geometry: P={} M={} bits/reg={}", ewm_core::P, ewm_core::M, ewm_core::BITS_PER_REG);


      tid0: hash=2892634914804110692 pos=(356, 0)  matches=true
    tid671: hash=15742022249931256727 pos=(919, 2)  matches=true
  tid18308: hash=14640161886912736849 pos=(593, 0)  matches=true
     hello: hash=14688674573012802306 pos=(770, 1)  matches=true
geometry: P=10 M=1024 bits/reg=32


InLUT recovery is the materialization morphism. The bridge golden model
(`InLut` + `slice_positions_with`) must recover exactly what the vendored
`materialize_inlut` recovers, for the same `tid{n}` inscriptions.


In [5]:
let ids = [1169u32, 412, 22117, 995, 2746, 9384, 27140, 44, 16326];
let tokens: Vec<Vec<u8>> = ids.iter().map(|&n| token_in_bytes(n)).collect();

// Vendored reference.
let vendored_lut = TokenLUT::from_tokens(tokens.iter());
let hll = HLLSet::from_tokens(tokens.iter());
let vendored_out = materialize_inlut(&hll, &vendored_lut).flat_tokens();

// Bridge golden model.
let mut bridge_lut = InLut::new();
for &id in &ids { bridge_lut.insert(token_in_bytes(id)); }
let positions: Vec<(u32, u32)> = ids.iter().map(|&id| token_to_position(&token_in_bytes(id))).collect();
let bridge_out = slice_positions_with(&positions, &bridge_lut, parse_token_id).ids;

println!("vendored recovered {} tokens, bridge recovered {} — equal: {}",
    vendored_out.len(), bridge_out.len(), vendored_out.len() == bridge_out.len());
println!("bridge ids: {:?}", bridge_out);


vendored recovered 9 tokens, bridge recovered 9 — equal: true
bridge ids: [44, 412, 995, 1169, 2746, 9384, 16326, 22117, 27140]


---
## 2. The cortex black-box as an `ewm-dsl` pipeline

The cortex pipeline (`tokens → hash → LUT → HLLSet → materialize → gate_TF → restored`)
is declared once in the DSL and run over the bridge's golden module registry.
The same declaration will drive the simulated FPGA backend when bridge Phase E lands.


In [6]:
let pipeline: ewm_dsl::Pipeline = cortex_pipeline();
println!("nodes:");
for n in pipeline.nodes() {
    println!("  node {}: {:?} ({})", n.id, n.kind, n.label.as_deref().unwrap_or(""));
}
println!("edges:");
for e in pipeline.edges() {
    println!("  {}:{} -> {}:{}", e.from.0, e.from.1, e.to.0, e.to.1);
}


nodes:
  node 0: Slice (materialize)
  node 1: Gate (gate_TF)
edges:
  0:0 -> 1:0


()

End-to-end golden run. The LUT is ungated (novel ids are registered before
slicing); only the output passes `gate_TF`. Out-of-vocab ids are reported as
leaks, never hidden.


In [7]:
let vocab = [0u32, 1, 44, 464, 671, 1169, 16326, 18308, 27140];
let doc_ids = [1169u32, 44, 18308];

let fpga = run_cortex_pipeline(&doc_ids, &vocab, &vocab).expect("golden run");
println!("document ids : {:?}", doc_ids);
println!("materialized : {:?}", fpga.materialized_ids);
println!("restored     : {:?}  leaks={:?}  ok={}", fpga.restored_ids, fpga.leaks, fpga.ok());

// Vendored CortexPipeline must agree.
let mut reference = CortexPipeline::new();
let vocab_bytes: Vec<Vec<u8>> = vocab.iter().map(|&n| token_in_bytes(n)).collect();
reference.set_gate(vocab_bytes.iter());
let doc_bytes: Vec<Vec<u8>> = doc_ids.iter().map(|&n| token_in_bytes(n)).collect();
let result = reference.process(&doc_bytes);
println!("vendored restored: {:?}  leaks={:?}", result.restored_ids, result.leaks);
let mut ours: Vec<String> = fpga.restored_ids.iter().map(|&n| format!("tid{n}")).collect();
ours.sort();
let mut theirs: Vec<String> = result.restored_ids.iter().map(|t| String::from_utf8_lossy(t).to_string()).collect();
theirs.sort();
println!("restored sets match: {}", ours == theirs);


document ids : [1169, 44, 18308]
materialized : [44, 1169, 18308]
restored     : [44, 1169, 18308]  leaks=[]  ok=true
vendored restored: [[116, 105, 100, 52, 52], [116, 105, 100, 49, 56, 51, 48, 56], [116, 105, 100, 49, 49, 54, 57]]  leaks=[]
restored sets match: true


In [8]:
// OOV demo: tid9 is outside the decoder vocabulary.
let oov = run_cortex_pipeline(&[0u32, 9], &[0u32, 1, 2, 3], &[0u32, 1, 2, 3]).expect("golden");
println!("materialized={:?}  restored={:?}  leaks={:?}", oov.materialized_ids, oov.restored_ids, oov.leaks);
println!("the LUT is ungated (tid9 materialized); only the output gate filters it.");


materialized=[0, 9]  restored=[0]  leaks=[9]
the LUT is ungated (tid9 materialized); only the output gate filters it.


---
## 3. Bridge Phase D nodes

`SliceModule` is now dual-encoding: `le` (notebook-08, 4-byte LE) and `tid`
(nanoLM / ewm-cortex). `GroundModule` carries the nanoLM ContextMatrix + the
τ/ρ grounding report.


In [9]:
fn bits_of(ids: &[u32], le: bool) -> Vec<u32> {
    ids.iter().map(|&id| {
        let bytes = if le { token_in_bytes_le(id).to_vec() } else { token_in_bytes(id) };
        let (reg, tz) = token_to_position(&bytes);
        reg * 32 + tz
    }).collect()
}

// notebook-08 inscription (4-byte LE).
let mut le_slice = SliceModule::new(0);
le_slice.load_config(&[("lut_ids".to_string(), "44,262,412".to_string())]).unwrap();
let mut le_in = vec![Stream { data: Packet { ids: bits_of(&[44, 262, 412], true), values: vec![] }, valid: true, ready: false }];
println!("LE   slice -> {:?}", le_slice.step(&mut le_in).unwrap()[0].data.ids);

// nanoLM / ewm-cortex inscription (tid{n}).
let mut tid_slice = SliceModule::new(0);
tid_slice.load_config(&[("encoding".to_string(), "tid".to_string()), ("lut_ids".to_string(), "0,671,18308".to_string())]).unwrap();
let mut tid_in = vec![Stream { data: Packet { ids: bits_of(&[0, 671, 18308], false), values: vec![] }, valid: true, ready: false }];
println!("tid  slice -> {:?}", tid_slice.step(&mut tid_in).unwrap()[0].data.ids);


LE   slice -> [44, 262, 412]
tid  slice -> [0, 671, 18308]


In [10]:
fn feed(g: &mut GroundModule, ids: Vec<u32>) -> (Packet, Packet) {
    let mut inputs = vec![Stream { data: Packet { ids, values: vec![] }, valid: true, ready: false }];
    let out = g.step(&mut inputs).unwrap();
    (out[0].data.clone(), out[1].data.clone())
}

let mut ground = GroundModule::new(0);
ground.load_config(&[("lut_ids".to_string(), "100,101,102,103".to_string())]).unwrap();

let (prior, report) = feed(&mut ground, vec![100, 101, 102]);
println!("prior : nodes={:?} bias={:?}", prior.ids, prior.values);
println!("report: tau={} rho={} srho={} r_link={} flagged={:?}",
    report.values[0] as f64 / 1e6, report.values[1] as f64 / 1e6,
    report.values[2] as f64 / 1e6, report.values[3], report.ids);

let (_, report2) = feed(&mut ground, vec![100, 999]);
println!("novel id 999 -> flagged={:?} tau={} rho={}",
    report2.ids, report2.values[0] as f64 / 1e6, report2.values[1] as f64 / 1e6);


prior : nodes=[100, 101, 102] bias=[1, 1, 0]
report: tau=1 rho=0 srho=0 r_link=0 flagged=[]
novel id 999 -> flagged=[999] tau=0.5 rho=0.5


---
## 4. LUT-view v1 — the content-addressed active vocabulary

`lut_view::LutView` is a vector of `sha1(token)` hashes identified by its own
SHA-1 digest (`v:<40-hex>`). It is the in-memory cache of the working
vocabulary: recomputed from `tokens_out` on context change and replaced only
when the digest changes.


In [11]:
let view = view_from_tokens(&fpga.materialized_ids);
println!("LUT-view: {} leaves, key = {}", view.len(), view.key());
println!("first leaves: {:?}", &view.hashes()[..view.len().min(3)]);

// Same context again → the digest is unchanged → no replacement (no churn).
let again = run_cortex_pipeline(&doc_ids, &vocab, &vocab).expect("golden");
let again_view = view_from_tokens(&again.materialized_ids);
println!("same doc -> refresh: {:?}", view.refresh(again_view.hashes()).map(|v| v.key()));

// A new token enters the context → the view is replaced with a new instance.
let bigger = run_cortex_pipeline(&[1169, 44, 18308, 671], &vocab, &vocab).expect("golden");
let bigger_view = view_from_tokens(&bigger.materialized_ids);
let next = view.refresh(bigger_view.hashes()).expect("vocabulary grew");
println!("new token -> replaced: {} leaves, key = {}", next.len(), next.key());
println!("digests differ: {}", view.digest() != next.digest());


LUT-view: 3 leaves, key = v:4e135d9476cac6c9e709077bfa239a32d4078275
first leaves: [[101, 210, 58, 229, 0, 116, 182, 46, 248, 78, 86, 191, 190, 177, 189, 212, 22, 116, 66, 155], [254, 152, 246, 236, 30, 162, 155, 162, 79, 156, 215, 2, 38, 234, 241, 151, 86, 62, 154, 47], [121, 194, 106, 151, 29, 31, 179, 137, 140, 4, 247, 53, 7, 196, 181, 41, 118, 96, 124, 214]]
same doc -> refresh: None
new token -> replaced: 4 leaves, key = v:cc6039a0cc3e422c2c4b849dff8b6cb5244a37f5
digests differ: true


---
## Summary

- The vendored reference and the bridge agree bit-exactly (hashing, InLUT, gate).
- The cortex black-box runs end-to-end as an `ewm-dsl` `Slice → Gate` pipeline
  over the bridge's golden modules — the sim backend swaps in later without
  touching the declaration.
- `SliceModule` handles both golden inscriptions; `GroundModule` emits the
  nanoLM prior and the τ/ρ report.
- `LutView` is the disposable, SHA-1-addressable active-vocabulary cache:
  stable contexts cause zero replacements, new tokens cause exactly one.

Upgrade path (deliberately not built): canonical sorted-set form, Merkle tree
with proofs/diff/merge, per-LUT provenance. See `docs/LUT_VIEW.md`.
